<a href="https://colab.research.google.com/github/SivaSam10/SivaSam/blob/main/Movie_Recommendation_RAG_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Personalized Movie Recommendation Engine (RAG-based)
### Built using Sentence Transformers + LlamaIndex + LangChain + Google Gemini API

**College Assignment Project — Beginner Friendly**

This notebook builds a **content-based movie recommendation system** that understands natural-language requests like *"Recommend action movies like John Wick"* and returns the **Top 5 matching movies** with genre, release year, rating, and a short description.

**Pipeline overview:**
1. Load a movie dataset (TMDB-style columns: title, genres, overview, keywords, cast, release year, rating).
2. Preprocess: handle missing values, select relevant columns, combine them into one text field per movie.
3. Generate semantic embeddings for each movie using **Sentence Transformers**.
4. Index the embeddings with **LlamaIndex** using a **FAISS** vector store.
5. Use **LangChain** to define the recommendation prompt and manage the RAG chain.
6. Use **Google Gemini API** to interpret the user's request and generate a clean Top-5 recommendation list from the retrieved candidates.
7. Test with sample questions and see the formatted output.


## Step 1 — Install Required Libraries

- `pandas`, `numpy`, `scikit-learn` — data handling and TF-IDF alternative
- `sentence-transformers` — semantic embeddings for movie content
- `llama-index` (+ FAISS & HuggingFace embedding integrations) — vector indexing & retrieval
- `langchain` / `langchain-google-genai` — prompt & chain management
- `faiss-cpu` — vector similarity search
- `google-generativeai` — Gemini LLM access

This may take 1-2 minutes on first run.


In [ ]:
# Install all required packages (Colab-friendly, quiet mode)
!pip install -q pandas numpy scikit-learn sentence-transformers \
    langchain langchain-google-genai langchain-community \
    llama-index llama-index-llms-gemini llama-index-embeddings-huggingface \
    faiss-cpu google-generativeai chromadb


## Step 2 — Set Up Your Google Gemini API Key

Get a **free** key from https://aistudio.google.com/app/apikey. `getpass` keeps it hidden and out of the saved notebook.


In [ ]:
import os
from getpass import getpass

os.environ['GOOGLE_API_KEY'] = getpass('Enter your Google Gemini API key: ')

# ---- OPTIONAL: OpenAI fallback instead of Gemini ----
# os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')
# USE_OPENAI = True   # then swap the LLM in Step 6 for llama_index.llms.openai.OpenAI
USE_OPENAI = False


## Step 3 — Import Libraries


In [ ]:
import pandas as pd
import numpy as np

# --- LlamaIndex core (indexing & querying) ---
from llama_index.core import VectorStoreIndex, Document, Settings, StorageContext
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore

# --- Embeddings (sentence-transformers via HuggingFace wrapper) ---
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# --- LLM: Google Gemini ---
from llama_index.llms.gemini import Gemini

# --- LangChain: prompt template management ---
from langchain.prompts import PromptTemplate

import faiss
print('Libraries imported successfully ✅')


## Step 4 — Load the Movie Dataset

This notebook ships with a **self-contained sample dataset** (20 movies) styled after the TMDB dataset (`title`, `genres`, `overview`, `keywords`, `cast`, `release_year`, `rating`), so it runs end-to-end with no external downloads or Kaggle authentication needed.

**To use the real TMDB 5000 Movies dataset instead:**
1. Download `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` from Kaggle.
2. Upload them in Colab (`Files` panel → upload), then load with:
```python
movies_df = pd.read_csv('tmdb_5000_movies.csv')
credits_df = pd.read_csv('tmdb_5000_credits.csv')
# merge on title, then parse the JSON-like genres/keywords/cast columns with ast.literal_eval
```
The preprocessing steps below work the same either way, as long as the final DataFrame has the columns used in Step 5.


In [ ]:
# Self-contained sample movie dataset (TMDB-style schema).
# In a real project, replace this with movies_df loaded from the TMDB CSV files (see markdown above).
data = [
 {'title': 'John Wick', 'genres': 'Action, Thriller', 'overview': 'An ex-hitman comes out of retirement to track down the gangsters that killed his dog and took everything from him.', 'keywords': 'assassin, revenge, hitman, dog', 'cast': 'Keanu Reeves, Michael Nyqvist', 'release_year': 2014, 'rating': 7.4},
 {'title': 'John Wick: Chapter 2', 'genres': 'Action, Thriller, Crime', 'overview': 'John Wick is forced back into the assassin world to repay a debt, facing a powerful crime syndicate.', 'keywords': 'assassin, revenge, crime syndicate', 'cast': 'Keanu Reeves, Common', 'release_year': 2017, 'rating': 7.5},
 {'title': 'Extraction', 'genres': 'Action, Thriller', 'overview': 'A black-market mercenary who has nothing to lose is hired to rescue the kidnapped son of an imprisoned crime lord.', 'keywords': 'mercenary, rescue, gunfights', 'cast': 'Chris Hemsworth', 'release_year': 2020, 'rating': 6.7},
 {'title': 'The Equalizer', 'genres': 'Action, Crime, Thriller', 'overview': 'A former black ops commando uses his particular set of skills to help those in need, taking on the Russian mob.', 'keywords': 'vigilante, ex-agent, revenge', 'cast': 'Denzel Washington', 'release_year': 2014, 'rating': 7.2},
 {'title': 'Mad Max: Fury Road', 'genres': 'Action, Adventure, Sci-Fi', 'overview': 'In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler in search of her homeland with the aid of a group of female prisoners.', 'keywords': 'post-apocalyptic, chase, wasteland', 'cast': 'Tom Hardy, Charlize Theron', 'release_year': 2015, 'rating': 8.1},
 {'title': 'Paddington 2', 'genres': 'Comedy, Family, Adventure', 'overview': 'Paddington bear tries to buy a special present for his aunt but it is stolen, leading to a heartwarming and hilarious adventure.', 'keywords': 'bear, family, heartwarming, comedy', 'cast': 'Hugh Grant, Ben Whishaw', 'release_year': 2017, 'rating': 7.8},
 {'title': 'The Incredibles', 'genres': 'Animation, Family, Comedy, Action', 'overview': 'A family of undercover superheroes tries to live a quiet suburban life while battling a new villainous threat.', 'keywords': 'superhero, family, animation', 'cast': 'Craig T. Nelson, Holly Hunter', 'release_year': 2004, 'rating': 8.0},
 {'title': 'Sing 2', 'genres': 'Animation, Comedy, Family, Musical', 'overview': 'Buster Moon and his cast of animal performers must persuade reclusive rock star Clay Calloway to join them for a new show.', 'keywords': 'music, animation, family, comedy', 'cast': 'Matthew McConaughey, Reese Witherspoon', 'release_year': 2021, 'rating': 7.4},
 {'title': 'Dune', 'genres': 'Sci-Fi, Adventure, Drama', 'overview': 'A noble family becomes embroiled in a war for control over the galaxy\'s most valuable asset on a dangerous desert planet.', 'keywords': 'desert planet, prophecy, empire', 'cast': 'Timothee Chalamet, Zendaya', 'release_year': 2021, 'rating': 8.0},
 {'title': 'Dune: Part Two', 'genres': 'Sci-Fi, Adventure, Drama', 'overview': 'Paul Atreides unites with the Fremen to seek revenge against the conspirators who destroyed his family, while facing a choice between love and the fate of the universe.', 'keywords': 'desert planet, revenge, prophecy', 'cast': 'Timothee Chalamet, Zendaya', 'release_year': 2024, 'rating': 8.5},
 {'title': 'Everything Everywhere All at Once', 'genres': 'Sci-Fi, Comedy, Adventure', 'overview': 'An exhausted laundromat owner discovers she must connect with parallel universe versions of herself to save existence.', 'keywords': 'multiverse, family, absurdist comedy', 'cast': 'Michelle Yeoh, Ke Huy Quan', 'release_year': 2022, 'rating': 7.8},
 {'title': 'Spider-Man: Into the Spider-Verse', 'genres': 'Animation, Action, Sci-Fi, Family', 'overview': 'Teen Miles Morales becomes Spider-Man and teams up with alternate-universe Spider-people to stop a dangerous threat.', 'keywords': 'superhero, multiverse, animation, family', 'cast': 'Shameik Moore', 'release_year': 2018, 'rating': 8.4},
 {'title': 'Superbad', 'genres': 'Comedy', 'overview': 'Two co-dependent high school seniors are forced to deal with separation anxiety after their plan to stage a booze-soaked party goes awry.', 'keywords': 'high school, friendship, comedy', 'cast': 'Jonah Hill, Michael Cera', 'release_year': 2007, 'rating': 7.6},
 {'title': 'The Grand Budapest Hotel', 'genres': 'Comedy, Adventure, Drama', 'overview': 'A legendary concierge at a famous European hotel and his loyal lobby boy become involved in a strange caper involving a stolen painting.', 'keywords': 'hotel, caper, quirky comedy', 'cast': 'Ralph Fiennes', 'release_year': 2014, 'rating': 8.1},
 {'title': 'Interstellar', 'genres': 'Sci-Fi, Adventure, Drama', 'overview': 'A team of explorers travels through a wormhole in space in an attempt to ensure humanity\'s survival as Earth becomes uninhabitable.', 'keywords': 'space travel, wormhole, survival', 'cast': 'Matthew McConaughey, Anne Hathaway', 'release_year': 2014, 'rating': 8.6},
 {'title': 'Arrival', 'genres': 'Sci-Fi, Drama, Mystery', 'overview': 'A linguist is recruited by the military to communicate with alien visitors after mysterious spacecraft touch down around the world.', 'keywords': 'aliens, linguistics, first contact', 'cast': 'Amy Adams', 'release_year': 2016, 'rating': 7.9},
 {'title': 'Knives Out', 'genres': 'Mystery, Comedy, Crime', 'overview': 'A detective investigates the death of a patriarch of an eccentric, combative family, uncovering a web of lies and motives.', 'keywords': 'whodunit, family, detective', 'cast': 'Daniel Craig, Chris Evans', 'release_year': 2019, 'rating': 7.9},
 {'title': 'Toy Story 4', 'genres': 'Animation, Family, Comedy, Adventure', 'overview': 'Woody and the gang embark on a road trip with new toy Forky, leading Woody to rediscover his purpose in life.', 'keywords': 'toys, animation, road trip, family', 'cast': 'Tom Hanks, Tim Allen', 'release_year': 2019, 'rating': 7.7},
 {'title': 'Nobody', 'genres': 'Action, Thriller, Crime', 'overview': 'An unassuming family man with a mysterious past unleashes a ferocious set of skills to take down a group of criminals after his home is targeted.', 'keywords': 'hidden past, revenge, ordinary hero', 'cast': 'Bob Odenkirk', 'release_year': 2021, 'rating': 7.4},
 {'title': 'Klaus', 'genres': 'Animation, Family, Comedy, Adventure', 'overview': 'A selfish postman and a reclusive toymaker strike an unlikely friendship that brings joy to a frozen, feuding town.', 'keywords': 'christmas, animation, friendship, family', 'cast': 'Jason Schwartzman', 'release_year': 2019, 'rating': 8.2},
]

movies_df = pd.DataFrame(data)
print(f'Loaded {len(movies_df)} movies ✅')
movies_df.head()


## Step 5 — Preprocess the Data

1. Handle missing values (drop rows missing a title/overview, fill other gaps).
2. Select the relevant columns: `title`, `genres`, `overview`, `keywords`, `cast`, `release_year`, `rating`.
3. Combine the text fields into a single `content` field per movie — this is what gets embedded for semantic search.


In [ ]:
# 1. Handle missing values
movies_df = movies_df.dropna(subset=['title', 'overview'])          # must have a title & overview
movies_df['genres'] = movies_df['genres'].fillna('Unknown')
movies_df['keywords'] = movies_df['keywords'].fillna('')
movies_df['cast'] = movies_df['cast'].fillna('Unknown')
movies_df['rating'] = movies_df['rating'].fillna(movies_df['rating'].mean())
movies_df['release_year'] = movies_df['release_year'].fillna(0).astype(int)

# 2. Select relevant columns (already present, but this mirrors what you'd do with a raw TMDB CSV)
relevant_cols = ['title', 'genres', 'overview', 'keywords', 'cast', 'release_year', 'rating']
movies_df = movies_df[relevant_cols].reset_index(drop=True)

# 3. Combine text fields into one 'content' string per movie for embedding
movies_df['content'] = (
    'Title: ' + movies_df['title'] +
    '. Genres: ' + movies_df['genres'] +
    '. Overview: ' + movies_df['overview'] +
    '. Keywords: ' + movies_df['keywords'] +
    '. Cast: ' + movies_df['cast'] +
    '. Release Year: ' + movies_df['release_year'].astype(str) +
    '. Rating: ' + movies_df['rating'].astype(str)
)

print('Preprocessing complete ✅')
movies_df[['title', 'content']].head(2)


## Step 6 — Generate Embeddings

We use **Sentence Transformers** (`all-MiniLM-L6-v2`) to turn each movie's `content` text into a semantic embedding vector, so the system can match *meaning* (e.g. "gritty revenge action movie") rather than just keyword overlap.

*(A TF-IDF alternative using `scikit-learn` is included as a comment if you'd like a lighter-weight, non-neural option for comparison.)*


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the sentence-transformers embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Generate an embedding for every movie's combined content (used later to sanity-check the index)
movie_embeddings = embedder.encode(movies_df['content'].tolist(), show_progress_bar=True)
print('Embeddings shape:', movie_embeddings.shape)   # (num_movies, 384)

# ---- OPTIONAL: TF-IDF alternative (classic, non-neural approach) ----
# from sklearn.feature_extraction.text import TfidfVectorizer
# tfidf = TfidfVectorizer(stop_words='english')
# tfidf_matrix = tfidf.fit_transform(movies_df['content'])


## Step 7 — Index the Embeddings with LlamaIndex + FAISS

1. Wrap each movie as a LlamaIndex `Document`, attaching genre/year/rating as **metadata** (so it can be displayed later without relying on the LLM to recall exact numbers).
2. Configure the same `sentence-transformers` model as LlamaIndex's embedding model.
3. Build a FAISS vector store and index all movie documents for fast semantic retrieval.


In [ ]:
# 1. Wrap each movie as a Document with rich metadata for accurate display later
documents = []
for _, row in movies_df.iterrows():
    doc = Document(
        text=row['content'],
        metadata={
            'title': row['title'],
            'genres': row['genres'],
            'release_year': int(row['release_year']),
            'rating': float(row['rating']),
            'overview': row['overview'],
        },
    )
    documents.append(doc)

# 2. Configure embedding model + Gemini LLM as LlamaIndex defaults
embed_model = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')
llm = Gemini(model='models/gemini-1.5-flash', api_key=os.environ['GOOGLE_API_KEY'])

Settings.embed_model = embed_model
Settings.llm = llm
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)

# 3. Build a FAISS vector store sized for the MiniLM-L6-v2 embedding dimension (384)
faiss_dim = 384
faiss_index = faiss.IndexFlatL2(faiss_dim)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Build the index — embeds each movie document and stores the vectors in FAISS
movie_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

print('Movie vector index built successfully ✅')


## Step 8 — Build the Recommendation Prompt with LangChain

LangChain's `PromptTemplate` defines exactly how Gemini should turn **retrieved candidate movies** into a clean **Top-5 recommendation list**, applying any constraints in the user's question (genre, year, similarity to a named movie, etc.).


In [ ]:
recommendation_prompt = PromptTemplate(
    input_variables=['context_str', 'query_str'],
    template=(
        'You are a movie recommendation assistant.\n'
        'Below is a list of candidate movies (retrieved by semantic search), each with its genres, '
        'release year, rating, and overview.\n\n'
        'Candidates:\n{context_str}\n\n'
        'User request: {query_str}\n\n'
        'Task: Select the TOP 5 candidates that best match the user request '
        '(respect any genre, year, or similarity constraints mentioned). '
        'If fewer than 5 good matches exist, return only the good matches.\n'
        'Format each recommendation EXACTLY like this:\n'
        '1. <Title> (<Release Year>) - Genre: <Genres> - Rating: <Rating>/10\n'
        '   Description: <one short sentence summary>\n\n'
        'Only use the candidates provided above — do not invent movies.'
    ),
)

from llama_index.core import PromptTemplate as LlamaPromptTemplate
llama_rec_prompt = LlamaPromptTemplate(recommendation_prompt.template)

print('Recommendation prompt ready ✅')


## Step 9 — Build the RAG Recommendation Engine

1. `similarity_top_k=10` retrieves a wider candidate pool via semantic search on the FAISS index.
2. Our custom prompt asks Gemini to pick and format the best 5 from those candidates, applying constraints from the user's natural-language request.
3. This is the core **Retrieval-Augmented Generation** loop: retrieve → ground the LLM in real data → generate a structured answer.


In [ ]:
# Build the RAG query engine on top of the movie index
recommendation_engine = movie_index.as_query_engine(
    similarity_top_k=10,               # cast a wide net, then let the LLM pick the best 5
    text_qa_template=llama_rec_prompt,
)

def recommend_movies(user_query: str) -> str:
    '''
    Full recommendation pipeline:
    1. Embed the user's natural-language request
    2. Retrieve the most semantically similar movies from FAISS
    3. Ask Gemini to select & format the Top 5 based on the request's constraints
    4. Return the formatted recommendation list
    '''
    response = recommendation_engine.query(user_query)
    return str(response)

print('Recommendation engine ready ✅')


## Step 10 — Test with Sample Questions

Let's try the exact example questions from the assignment, plus two more.


In [ ]:
sample_queries = [
    'Recommend action movies like John Wick.',
    'Suggest family-friendly comedy movies.',
    'Recommend sci-fi movies released after 2020.',
    'I want a witty mystery movie with an ensemble cast.',
    'Suggest animated movies about family and adventure.',
]

for i, q in enumerate(sample_queries, start=1):
    print(f'User Query {i}: {q}')
    print(recommend_movies(q))
    print('=' * 90)


## Expected Output (example)

```
User Query 1: Recommend action movies like John Wick.
1. John Wick: Chapter 2 (2017) - Genre: Action, Thriller, Crime - Rating: 7.5/10
   Description: An ex-hitman is pulled back into the assassin underworld to repay a debt.
2. Nobody (2021) - Genre: Action, Thriller, Crime - Rating: 7.4/10
   Description: A mild-mannered family man reveals a deadly hidden past after his home is targeted.
3. Extraction (2020) - Genre: Action, Thriller - Rating: 6.7/10
   Description: A mercenary undertakes a dangerous mission to rescue a kidnapped boy.
4. The Equalizer (2014) - Genre: Action, Crime, Thriller - Rating: 7.2/10
   Description: A former black-ops agent takes on the Russian mob to protect the vulnerable.
5. Mad Max: Fury Road (2015) - Genre: Action, Adventure, Sci-Fi - Rating: 8.1/10
   Description: A rebel woman leads a high-octane escape across a post-apocalyptic wasteland.
==========================================================================================
User Query 3: Recommend sci-fi movies released after 2020.
1. Dune: Part Two (2024) - Genre: Sci-Fi, Adventure, Drama - Rating: 8.5/10
   Description: Paul Atreides unites with the Fremen to seek revenge and decide the fate of the universe.
2. Everything Everywhere All at Once (2022) - Genre: Sci-Fi, Comedy, Adventure - Rating: 7.8/10
   Description: A woman must connect with parallel-universe versions of herself to save existence.
3. Dune (2021) - Genre: Sci-Fi, Adventure, Drama - Rating: 8.0/10
   Description: A noble family is drawn into a war over a desert planet's most valuable resource.
...
```

> Exact wording is generated live by Gemini and will vary slightly between runs, but titles/years/ratings are pulled from your indexed dataset and should stay accurate and grounded.


## Step 11 (Optional) — Interactive Recommendation Loop

Run this cell to ask your own questions interactively. Type `exit` to stop.


In [ ]:
# Optional interactive loop — run this cell and type your own movie requests
while True:
    user_input = input('You: ')
    if user_input.strip().lower() in ['exit', 'quit']:
        print('Recommender: Enjoy the movie! 🍿')
        break
    print('\nRecommender:')
    print(recommend_movies(user_input))
    print()


## Notes & Possible Extensions

- **Use the real TMDB dataset:** Load `tmdb_5000_movies.csv` + `tmdb_5000_credits.csv`, merge on title, parse the JSON-encoded `genres`/`keywords`/`cast` columns with `ast.literal_eval`, then feed the result into Step 5 unchanged.
- **Swap FAISS for ChromaDB:** Replace the `FaissVectorStore` block in Step 7 with `llama_index.vector_stores.chroma.ChromaVectorStore` for a persistent, disk-backed vector database that survives notebook restarts.
- **Add collaborative filtering:** Combine this content-based approach with a user-ratings-based collaborative filter (e.g. via `surprise` or matrix factorization on MovieLens ratings) for a hybrid recommender.
- **Switch to OpenAI:** Set `USE_OPENAI = True` in Step 2, then replace the Gemini LLM in Step 7 with `from llama_index.llms.openai import OpenAI` and `llm = OpenAI(model='gpt-4o-mini')`.
- **Add a UI:** Wrap `recommend_movies()` with `gradio` (`!pip install gradio`) for a simple shareable web app — a nice demo addition for a project presentation.
